In [91]:
import torch
import faiss
import numpy as np
from PIL import Image

from ultralytics import YOLO
import open_clip

from pathlib import Path
import os

import sys
from pathlib import Path

# 프로젝트 루트 추가
ROOT = Path().resolve().parent   # fastapi-agent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# 클래스만 import
from config.settings import Settings

# 인스턴스 생성 (env 자동 로드)
settings = Settings()

# print("Storage path:", settings.STORAGE_PATH)
# print("Upload path:", settings.UPLOAD_PATH)
# print("Image path:", settings.IMAGE_PATH)


In [92]:
yolo = YOLO("yolov8n.pt")

In [93]:
device = "cuda" if torch.cuda.is_available() else "cpu"

clip_model, _, preprocess = open_clip.create_model_and_transforms(
    model_name="ViT-B-32",
    pretrained="laion2b_s34b_b79k"
)
clip_model = clip_model.to(device)
clip_model.eval()

CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
    (patch_dropout): Identity()
    (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): ModuleList(
        (0-11): 12 x ResidualAttentionBlock(
          (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (ls_1): Identity()
          (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=768, out_features=3072, bias=True)
            (gelu): GELU(approximate='none')
            (c_proj): Linear(in_features=3072, out_features=768, bias=True)
          )
          (ls_2): Identity()
        )
      )
    )
    (ln_post): LayerNorm((768,), eps=1e-05, elementwise_affine

In [94]:
def detect_and_crop(image_path):
    img = Image.open(image_path).convert("RGB")

    r = yolo(image_path)[0]

    if len(r.boxes) == 0:
        return None

    boxes = r.boxes.xyxy
    confs = r.boxes.conf
    clss = r.boxes.cls

    idx = confs.argmax().item()

    x1, y1, x2, y2 = map(int, boxes[idx])
    crop = img.crop((x1, y1, x2, y2))

    return {
        "crop": crop,
        "confidence": float(confs[idx]),
        "class_id": int(clss[idx]),
        "class_name": yolo.names[int(clss[idx])],
        "bbox": [x1, y1, x2, y2],
    }


    # crops = []
    # for box, conf, cls in zip(boxes, confs, clss):
    #     x1, y1, x2, y2 = map(int, box)
    #     crop = img.crop((x1, y1, x2, y2))
    #     crops.append({
    #         "crop": crop,
    #         "confidence": float(conf),
    #         "class_id": int(cls),
    #         "class_name": yolo.names[int(cls)],
    #     })

    # return crops

In [95]:
def get_clip_embedding(image: Image.Image):
    image_tensor = preprocess(image).unsqueeze(0).to(device)

    with torch.no_grad():
        embedding = clip_model.encode_image(image_tensor)

    embedding = embedding / embedding.norm(dim=1, keepdim=True)
    return embedding.cpu().numpy()[0]

In [96]:
embedding_dim = 512  # ViT-B-32
index = faiss.IndexFlatIP(embedding_dim)  # cosine similarity

In [97]:
IMAGE_DIR = ROOT / "storage" / "images" / "fruits"

image_paths = [
    p for p in IMAGE_DIR.glob("*")
    if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
]

id_map = []  # 어떤 이미지의 어떤 객체인지 추적

for path in image_paths:
    result = detect_and_crop(path)   # ← crops → result

    if result is None:
        print(f"없음: {path}")
        continue

    emb = get_clip_embedding(result["crop"])
    index.add(np.array([emb]))

    id_map.append({
        "image_path": str(path),
        "bbox": result["bbox"],
        "class_name": result["class_name"],
        "confidence": result["confidence"],
    })

print("등록된 객체 수:", index.ntotal)


image 1/1 C:\xampp\htdocs\www\projects\fastapi-agent\storage\images\fruits\0a734cbe4efc3caf56af9f8393ab4565.jpg: 640x640 2 bananas, 310.8ms
Speed: 26.4ms preprocess, 310.8ms inference, 3.4ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 C:\xampp\htdocs\www\projects\fastapi-agent\storage\images\fruits\41901b1c05226e911397129c6d743d9c.jpg: 640x384 9 apples, 3 oranges, 222.1ms
Speed: 6.1ms preprocess, 222.1ms inference, 2.9ms postprocess per image at shape (1, 3, 640, 384)

image 1/1 C:\xampp\htdocs\www\projects\fastapi-agent\storage\images\fruits\466e89bd090ad3feef8a8b0cf1aecdfd.jpg: 640x352 2 bananas, 173.6ms
Speed: 10.0ms preprocess, 173.6ms inference, 3.5ms postprocess per image at shape (1, 3, 640, 352)

image 1/1 C:\xampp\htdocs\www\projects\fastapi-agent\storage\images\fruits\582e9ac306a48e209e3138d5512e837b.jpg: 640x640 1 vase, 305.0ms
Speed: 12.4ms preprocess, 305.0ms inference, 4.0ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 C:\xampp\htdocs\www\pro

In [98]:
# ROOT = Path().resolve().parent

# # 로컬 이미지 경로
# image_path = ROOT / "storage" / "images" / "apple.jpg"

# result = detect_and_crop(image_path)
# print(result)

In [100]:
ROOT = Path().resolve().parent

# 로컬 이미지 경로
image_path = ROOT / "storage" / "images" / "grape.jpg"


query_image = Image.open(image_path).convert("RGB")
query_emb = get_clip_embedding(query_image)

D, I = index.search(np.array([query_emb]), k=5)

print("유사 이미지 결과:")
for score, idx in zip(D[0], I[0]):
    print(id_map[idx], score)

유사 이미지 결과:
{'image_path': 'C:\\xampp\\htdocs\\www\\projects\\fastapi-agent\\storage\\images\\fruits\\b9e533515c517e4b41aa281440269668.jpg', 'bbox': [86, 121, 403, 491], 'class_name': 'apple', 'confidence': 0.4360310137271881} 0.87101525
{'image_path': 'C:\\xampp\\htdocs\\www\\projects\\fastapi-agent\\storage\\images\\fruits\\587368bea8d187622a1e1d9ad24bdca0.jpg', 'bbox': [276, 311, 929, 1074], 'class_name': 'vase', 'confidence': 0.46569180488586426} 0.68166214
{'image_path': 'C:\\xampp\\htdocs\\www\\projects\\fastapi-agent\\storage\\images\\fruits\\8dfe8068aa2755cbe13ae938d9d7cf6f.jpg', 'bbox': [542, 46, 1143, 719], 'class_name': 'apple', 'confidence': 0.8136582970619202} 0.6520494
{'image_path': 'C:\\xampp\\htdocs\\www\\projects\\fastapi-agent\\storage\\images\\fruits\\41901b1c05226e911397129c6d743d9c.jpg', 'bbox': [259, 427, 592, 755], 'class_name': 'apple', 'confidence': 0.5494494438171387} 0.6279282
{'image_path': 'C:\\xampp\\htdocs\\www\\projects\\fastapi-agent\\storage\\images\\f